In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import sklearn as skl
# import sktime
import random
import os
# from sktime.forecasting.model_selection import temporal_train_test_split
# from sktime.utils.plotting import plot_series

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

%matplotlib inline
from matplotlib import rc
plt.rc('font', family='NanumBarunGothic')
# plt.rcParams['font.family'] ='NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(493) # Seed 고정

# 불러오기
train = pd.read_csv('./train.csv')
sales = pd.read_csv('./sales.csv')
product_info = pd.read_csv('./product_info.csv')
brand_keyword_cnt = pd.read_csv('./brand_keyword_cnt.csv')
sample_submission = pd.read_csv('./sample_submission.csv')
# price = pd.read_csv('./price.csv')

# **제품 정보 train 있는 것만 추출하기**

In [ ]:
# # 전자의 데이터프레임에서 브랜드 목록 추출
# existing_brands = train['제품'].unique()

# # 후자의 데이터프레임에서 전자의 데이터프레임에 있는 브랜드만 필터링하여 새로운 데이터프레임 생성
# filtered_df2 = product_info[product_info['제품'].isin(existing_brands)]

# # '제품' 컬럼을 기준으로 정렬
# filtered_df2_sorted = filtered_df2.sort_values(by='제품')
# product_info = filtered_df2_sorted

In [ ]:
# product_info

# **대/중/소/분류, 브랜드**

In [ ]:
# B002-03254-00002

In [ ]:
id_vars = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드']
train_melted = pd.melt(train, id_vars=id_vars, var_name='날짜', value_name='판매량')
train_melted['날짜'] = pd.to_datetime(train_melted['날짜'])
train_grouped = train_melted.groupby('ID')

groups = []
for i in tqdm(range(15889+1)):
    get_train = train_grouped.get_group(i).reset_index()
    get_train.drop('index', axis=1, inplace=True)
    groups.append(get_train)
train_vertical = pd.concat(groups, ignore_index=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
# a = train_vertical['대분류'].unique()
# print('대분류: ', a)

In [ ]:
# 대분류:  ['B002-C001-0002' 'B002-C001-0003' 'B002-C001-0001' 'B002-C001-0005', 'B002-C001-0004']

In [ ]:
# b = train_vertical[train_vertical['대분류'] == 'B002-C001-0003']['중분류'].unique()
# print('중분류: ', b)

In [ ]:
# c = train_vertical[train_vertical['중분류'] == 'B002-C002-0010']['소분류'].unique()
# print('소분류: ', c)

In [ ]:
product_info[product_info["제품특성"].str.contains("본품")]

# **#############################################**

In [ ]:
list_train = train_vertical[train_vertical['브랜드'] == 'B002-00236']['제품'].unique().tolist()

for product in list_train:
    result = product_info[product_info['제품'] == product]
    if not result.empty:
        print(result.to_string(index=False))
    else:
        print(f"No information found for product: {product}")

              제품                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               제품특성
B002-00236-00001 사용부위사용시간사용시간주요제품특징세부제품특징세부제품특징세부제품특징세부제품특징세부제품특징제품형태:페이스용, 피부타입피부타입세트수량주요제품특징주요제품특징

# **#############################################**

In [ ]:
list_train = train_vertical[train_vertical['소분류'] == 'B002-C003-0015']['제품'].unique().tolist()

for product in list_train:
    result = product_info[product_info['제품'] == product]
    if not result.empty:
        print(result.to_string(index=False))
    else:
        print(f"No information found for product: {product}")

In [ ]:
# list_train = train_vertical[train_vertical['소분류'] == 'B002-C003-0001']['제품'].unique().tolist()

# for product in list_train:
#     result = product_info[product_info['제품'] == product]
#     if not result.empty:
#         # '제품특성'에서 마지막 5글자를 추출
#         result['제품특성'] = result['제품특성'].str[-4:]
#         print(result.to_string(index=False))
#     else:
#         print(f"No information found for product: {product}")

In [ ]:
# # 마지막 5글자를 저장할 리스트
# last_five_characters = []

# list_train = train_vertical[train_vertical['소분류'] == 'B002-C003-0002']['제품'].unique().tolist()

# for product in list_train:
#     result = product_info[product_info['제품'] == product]
#     if not result.empty:
#         # '제품특성'에서 마지막 5글자를 추출하고 리스트에 추가
#         last_five_characters.extend(result['제품특성'].str[-5:].tolist())

# # 리스트에서 중복을 제거
# unique_last_five_characters = list(set(last_five_characters))

# # 결과 출력
# for item in unique_last_five_characters:
#     print(item)

In [ ]:
# train_vertical[train_vertical['제품']=='B002-00109-00005']

# **정리1**

In [ ]:
# <대분류: B002-C001-0001>

# 중분류: B002-C002-0001 > 소분류: B002-C003-0001:뷰티/다이어트, B002-C003-0002:비타민, B002-C003-0003:기능별영양제,  B002-C003-0004:단백질, B002-C003-0005:홍삼농축
# 중분류: B002-C002-0009 > 소분류: B002-C003-0045:유아용 스킨케어, B002-C003-0046:유아바디&샴푸, B002-C003-0047:유아선크림, B002-C003-0048:세탁세제(유아),  B002-C003-0049:유아칫솔, B002-C003-0051:일반분유

In [ ]:
# <대분류: B002-C001-0002>

# 중분류: B002-C002-0002 > 소분류: B002-C003-0006:방향제(차량용), B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제],  B002-C003-0008:[(향수), 로션, 세탁용세제], B002-C003-0009:수세미, B002-C003-0010:키친타월, B002-C003-0011:제습제
# 중분류: B002-C002-0003 > 소분류: B002-C003-0012:청소용세제, B002-C003-0013:주방세제(시트형), B002-C003-0014:섬유유연제, B002-C003-0015:세탁세제, B002-C003-0016:주방세제, B002-C003-0017:과탄산소다/베이킹소다, B002-C003-0018:표백제
# 중분류: B002-C002-0004 > 소분류: B002-C003-0019:가글, B002-C003-0020:전동칫솔, B002-C003-0021:치실/치간칫솔, B002-C003-0022:치약, B002-C003-0023:칫솔
# 중분류: B002-C002-0005 > 소분류: B002-C003-0024:화장지(롤), B002-C003-0025:물티슈, B002-C003-0026:각티슈, B002-C003-0027:생리대, B002-C003-0028:성인용 기저귀, B002-C003-0029:키친타월, B002-C003-0041:기저귀
# 중분류: B002-C002-0006 > 소분류: B002-C003-0030:면도기, B002-C003-0031:면도거품, B002-C003-0032:헨드케어(손비누), B002-C003-0033:전동면도기, B002-C003-0034:바디워시
# 중분류: B002-C002-0007 > 소분류: B002-C003-0035:샴푸, B002-C003-0036:에센스/트리트먼트, B002-C003-0037:컨디셔너(린스), B002-C003-0038:에센스/트리트먼트, B002-C003-0039:헤어스타일링, B002-C003-0040:에센스/트리트먼트
# 중분류: B002-C002-0009 > 소분류: B002-C003-0045:유아용 스킨케어, B002-C003-0046:유아바디&샴푸, B002-C003-0047:유아선크림, B002-C003-0048:세탁세제(유아), B002-C003-0049:유아칫솔, B002-C003-0051:일반분유

In [ ]:
# <대분류: B002-C001-0003>

# 중분류: B002-C002-0008 > 소분류: B002-C003-0042:젖병소독기, B002-C003-0043:젖병, B002-C003-0044: 유아식기(보관용기/식판/식기/컵/빨대)
# 중분류: B002-C002-0010 > 소분류: B002-C003-0050: 유아용 스킨케어

In [ ]:
# <대분류: B002-C001-0004>

# 중분류: B002-C002-0009 > 소분류: B002-C003-0045: 유아용 스킨케어, B002-C003-0046:유아바디&샴푸, B002-C003-0047:유아선크림, B002-C003-0048:세탁세제(유아), B002-C003-0049:유아칫솔, B002-C003-0051:일반분유

In [ ]:
# <대분류: B002-C001-0005>

# 중분류: B002-C002-0011 > 소분류: B002-C003-0052: 유아주스/차/떡벙(간식), B002-C003-0053: 이유식

# **정리2**

In [ ]:
# <대분류: B002-C001-0001>


# 중분류: B002-C002-0001 > 소분류: B002-C003-0001:뷰티/다이어트,
#                                  B002-C003-0002:비타민,
#                                  B002-C003-0003:기능별영양제,
#                                  B002-C003-0004:단백질,
#                                  B002-C003-0005:홍삼농축

# 중분류: B002-C002-0009 > 소분류: B002-C003-0045:유아용 스킨케어,
#                                  B002-C003-0046:유아바디&샴푸,
#                                  B002-C003-0047:유아선크림,
#                                  B002-C003-0048:세탁세제(유아),
#                                  B002-C003-0049:유아칫솔,
#                                  B002-C003-0051:일반분유

In [ ]:
# <대분류: B002-C001-0002>


# 중분류: B002-C002-0002 > 소분류: B002-C003-0006:방향제(차량용),
#                                  B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제],
#                                  B002-C003-0008:[(향수), 로션, 세탁용세제],
#                                  B002-C003-0009:수세미,
#                                  B002-C003-0010:키친타월,
#                                  B002-C003-0011:제습제

# 중분류: B002-C002-0003 > 소분류: B002-C003-0012:청소용세제,
#                                  B002-C003-0013:주방세제(시트형),
#                                  B002-C003-0014:섬유유연제,
#                                  B002-C003-0015:세탁세제,
#                                  B002-C003-0016:주방세제,
#                                  B002-C003-0017:과탄산소다/베이킹소다,
#                                  B002-C003-0018:표백제

# 중분류: B002-C002-0004 > 소분류: B002-C003-0019:가글,
#                                  B002-C003-0020:전동칫솔,
#                                  B002-C003-0021:치실/치간칫솔,
#                                  B002-C003-0022:치약,
#                                  B002-C003-0023:칫솔

# 중분류: B002-C002-0005 > 소분류: B002-C003-0024:화장지(롤),
#                                  B002-C003-0025:물티슈,
#                                  B002-C003-0026:각티슈,
#                                  B002-C003-0027:생리대,
#                                  B002-C003-0028:성인용 기저귀,
#                                  B002-C003-0029:키친타월,
#                                  B002-C003-0041:기저귀

# 중분류: B002-C002-0006 > 소분류: B002-C003-0030:면도기,
#                                  B002-C003-0031:면도거품,
#                                  B002-C003-0032:헨드케어(손비누),
#                                  B002-C003-0033:전동면도기,
#                                  B002-C003-0034:바디워시

# 중분류: B002-C002-0007 > 소분류: B002-C003-0035:샴푸,
#                                  B002-C003-0036:에센스/트리트먼트,
#                                  B002-C003-0037:컨디셔너(린스),
#                                  B002-C003-0038:에센스/트리트먼트,
#                                  B002-C003-0039:헤어스타일링,
#                                  B002-C003-0040:에센스/트리트먼트

# 중분류: B002-C002-0009 > 소분류: B002-C003-0045:유아용 스킨케어,
#                                  B002-C003-0046:유아바디&샴푸,
#                                  B002-C003-0047:유아선크림,
#                                  B002-C003-0048:세탁세제(유아),
#                                  B002-C003-0049:유아칫솔,
#                                  B002-C003-0051:일반분유

In [ ]:
# <대분류: B002-C001-0003>


# 중분류: B002-C002-0008 > 소분류: B002-C003-0042:젖병소독기,
#                                  B002-C003-0043:젖병,
#                                  B002-C003-0044: 유아식기(보관용기/식판/식기/컵/빨대)

# 중분류: B002-C002-0010 > 소분류: B002-C003-0050: 유아용 스킨케어

In [ ]:
# <대분류: B002-C001-0004>


# 중분류: B002-C002-0009 > 소분류: B002-C003-0045: 유아용 스킨케어,
#                                  B002-C003-0046:유아바디&샴푸,
#                                  B002-C003-0047:유아선크림,
#                                  B002-C003-0048:세탁세제(유아),
#                                  B002-C003-0049:유아칫솔,
#                                  B002-C003-0051:일반분유

In [ ]:
# <대분류: B002-C001-0005>


# 중분류: B002-C002-0011 > 소분류: B002-C003-0052: 유아주스/차/떡벙(간식),
#                                  B002-C003-0053: 이유식

# **나열**

In [ ]:
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축
# B002-C003-0006:방향제(차량용)
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]
# B002-C003-0008:[(향수), 로션, 세탁용세제]
# B002-C003-0009:수세미
# B002-C003-0010:키친타월
# B002-C003-0011:제습제
# B002-C003-0012:청소용세제
# B002-C003-0013:주방세제(시트형)
# B002-C003-0014:섬유유연제
# B002-C003-0015:세탁세제
# B002-C003-0016:주방세제
# B002-C003-0017:과탄산소다/베이킹소다
# B002-C003-0018:표백제
# B002-C003-0019:가글
# B002-C003-0020:전동칫솔
# B002-C003-0021:치실/치간칫솔
# B002-C003-0022:치약
# B002-C003-0023:칫솔
# B002-C003-0024:화장지(롤)
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈
# B002-C003-0027:생리대
# B002-C003-0028:성인용 기저귀
# B002-C003-0029:키친타월
# B002-C003-0030:면도기
# B002-C003-0031:면도거품
# B002-C003-0032:헨드케어(손비누)
# B002-C003-0033:전동면도기
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0039:헤어스타일링
# B002-C003-0040:에센스/트리트먼트
# B002-C003-0041:기저귀
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)
# B002-C003-0045:유아용 스킨케어
# B002-C003-0046:유아바디&샴푸
# B002-C003-0047:유아선크림
# B002-C003-0048:세탁세제(유아)
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어
# B002-C003-0051:일반분유
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식

# **LG 생활건강 몰 기준 큰 카테고리 - 폐기**

In [ ]:
## 건강기능식품
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축

## 아웃도어
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]

## 기타
# B002-C003-0008:[(향수), 로션, 세탁용세제]

## 세탁/주방/청소
# B002-C003-0009:수세미
# B002-C003-0010:키친타월
# B002-C003-0011:제습제
# B002-C003-0012:청소용세제
# B002-C003-0013:주방세제(시트형)
# B002-C003-0014:섬유유연제
# B002-C003-0015:세탁세제
# B002-C003-0048:세탁세제(유아)
# B002-C003-0016:주방세제
# B002-C003-0017:과탄산소다/베이킹소다
# B002-C003-0018:표백제

## 덴탈케어
# B002-C003-0019:가글
# B002-C003-0020:전동칫솔
# B002-C003-0021:치실/치간칫솔
# B002-C003-0022:치약
# B002-C003-0023:칫솔

## 일상용품
# B002-C003-0006:방향제(차량용)
# B002-C003-0024:화장지(롤)
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈
# B002-C003-0029:키친타월

## 위생용품
# B002-C003-0027:생리대
# B002-C003-0028:성인용 기저귀

## 바디
# B002-C003-0030:면도기
# B002-C003-0031:면도거품
# B002-C003-0033:전동면도기
# B002-C003-0032:헨드케어(손비누)
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0039:헤어스타일링
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0040:에센스/트리트먼트


## 유아동
# B002-C003-0041:기저귀
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)
# B002-C003-0045:유아용 스킨케어
# B002-C003-0046:유아바디&샴푸
# B002-C003-0047:유아선크림
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어
# B002-C003-0051:일반분유
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식

# **그냥 비슷한 품목들(그냥) - 폐기**


In [ ]:
## SET_A
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축

## SET_B
# B002-C003-0006:방향제(차량용)

## SET_C
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]

## SET_D
# B002-C003-0008:[(향수), 로션, 세탁용세제]

## SET_E
# B002-C003-0009:수세미

## SET_F
# B002-C003-0013:주방세제(시트형)
# B002-C003-0016:주방세제

## SET_G
# B002-C003-0010:키친타월
# B002-C003-0024:화장지(롤)
# B002-C003-0029:키친타월
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈

## SET_H
# B002-C003-0011:제습제

## SET_I
# B002-C003-0012:청소용세제
# B002-C003-0017:과탄산소다/베이킹소다

## SET_J
# B002-C003-0014:섬유유연제

## SET_K
# B002-C003-0015:세탁세제
# B002-C003-0018:표백제
# B002-C003-0048:세탁세제(유아)

## SET_L
# B002-C003-0019:가글
# B002-C003-0022:치약

## SET_M
# B002-C003-0021:치실/치간칫솔
# B002-C003-0023:칫솔
# B002-C003-0020:전동칫솔
# B002-C003-0049:유아칫솔

## SET_N
# B002-C003-0027:생리대

## SET_O
# B002-C003-0028:성인용 기저귀
# B002-C003-0041:기저귀

## SET_P
# B002-C003-0030:면도기
# B002-C003-0033:전동면도기

## SET_Q
# B002-C003-0031:면도거품

## SET_R
# B002-C003-0032:헨드케어(손비누)

## SET_S
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0046:유아바디&샴푸
# B002-C003-0045:유아용 스킨케어
# B002-C003-0050:유아용 스킨케어

## SET_T
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0040:에센스/트리트먼트

## SET_U
# B002-C003-0039:헤어스타일링

## SET_V
# B002-C003-0042:젖병소독기

## SET_W
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)

## SET_X
# B002-C003-0047:유아선크림

## SET_Y
# B002-C003-0051:일반분유

## SET_Z
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식

# **지속적/중간/약간하락/급하락 (판매량/매출) (상대성) - 폐기**

In [ ]:
## 지속적인 판매량/매출
# B002-C003-0010:키친타월
# B002-C003-0009:수세미
# B002-C003-0011:제습제
# B002-C003-0012:청소용세제
# B002-C003-0013:주방세제(시트형)
# B002-C003-0014:섬유유연제
# B002-C003-0015:세탁세제
# B002-C003-0016:주방세제
# B002-C003-0021:치실/치간칫솔
# B002-C003-0022:치약
# B002-C003-0023:칫솔
# B002-C003-0024:화장지(롤)
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈
# B002-C003-0027:생리대
# B002-C003-0028:성인용 기저귀
# B002-C003-0029:키친타월
# B002-C003-0030:면도기
# B002-C003-0031:면도거품
# B002-C003-0032:헨드케어(손비누)
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸

## 중간 포지션
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축
# B002-C003-0006:방향제(차량용)
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]
# B002-C003-0008:[(향수), 로션, 세탁용세제]
# B002-C003-0017:과탄산소다/베이킹소다
# B002-C003-0018:표백제
# B002-C003-0019:가글
# B002-C003-0020:전동칫솔
# B002-C003-0033:전동면도기
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0039:헤어스타일링
# B002-C003-0040:에센스/트리트먼트

# 약간 하락 하는 판매량/매출
# B002-C003-0041:기저귀
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)
# B002-C003-0045:유아용 스킨케어
# B002-C003-0046:유아바디&샴푸
# B002-C003-0047:유아선크림
# B002-C003-0048:세탁세제(유아)
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어

# 급 하락 하는 판매량/매출
# B002-C003-0051:일반분유
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식

# **대용량의 여부 (아직 안씀)**

In [ ]:
## 대용량 버전이 존재한다.
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축
# B002-C003-0010:키친타월
# B002-C003-0011:제습제
# B002-C003-0012:청소용세제
# B002-C003-0013:주방세제(시트형)
# B002-C003-0014:섬유유연제
# B002-C003-0015:세탁세제
# B002-C003-0016:주방세제
# B002-C003-0017:과탄산소다/베이킹소다
# B002-C003-0018:표백제
# B002-C003-0019:가글
# B002-C003-0022:치약
# B002-C003-0023:칫솔
# B002-C003-0024:화장지(롤)
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈
# B002-C003-0027:생리대
# B002-C003-0028:성인용 기저귀
# B002-C003-0029:키친타월
# B002-C003-0030:면도기
# B002-C003-0032:헨드케어(손비누)
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0040:에센스/트리트먼트
# B002-C003-0041:기저귀
# B002-C003-0045:유아용 스킨케어
# B002-C003-0046:유아바디&샴푸
# B002-C003-0047:유아선크림
# B002-C003-0048:세탁세제(유아)
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어
# B002-C003-0051:일반분유
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식


## 대용량 버전이 존재하지 않는다.
# B002-C003-0006:방향제(차량용)
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]
# B002-C003-0008:[(향수), 로션, 세탁용세제]
# B002-C003-0009:수세미
# B002-C003-0020:전동칫솔
# B002-C003-0021:치실/치간칫솔
# B002-C003-0031:면도거품
# B002-C003-0033:전동면도기
# B002-C003-0039:헤어스타일링
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)

# **리필분의 여부 (아직 안씀)**

In [ ]:
## 리필 분을 판다.(본품과 리필분이 존재)
# B002-C003-0006:방향제(차량용)
# B002-C003-0008:[(향수), 로션, 세탁용세제]
# B002-C003-0014:섬유유연제
# B002-C003-0015:세탁세제
# B002-C003-0016:주방세제
# B002-C003-0017:과탄산소다/베이킹소다
# B002-C003-0018:표백제
# B002-C003-0032:헨드케어(손비누)
# B002-C003-0048:세탁세제(유아)
# B002-C003-0030:면도기
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]
# B002-C003-0020:전동칫솔



## 리필 분을 팔지 않는다.(본품만 존재)
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축
# B002-C003-0009:수세미
# B002-C003-0010:키친타월
# B002-C003-0011:제습제
# B002-C003-0013:주방세제(시트형)
# B002-C003-0012:청소용세제
# B002-C003-0019:가글
# B002-C003-0021:치실/치간칫솔
# B002-C003-0022:치약
# B002-C003-0023:칫솔
# B002-C003-0024:화장지(롤)
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈
# B002-C003-0027:생리대
# B002-C003-0028:성인용 기저귀
# B002-C003-0029:키친타월
# B002-C003-0031:면도거품
# B002-C003-0033:전동면도기
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0039:헤어스타일링
# B002-C003-0040:에센스/트리트먼트
# B002-C003-0041:기저귀
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)
# B002-C003-0045:유아용 스킨케어
# B002-C003-0046:유아바디&샴푸
# B002-C003-0047:유아선크림
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어
# B002-C003-0051:일반분유
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식

# **함께 사용되는 것들로 묶음(애매한 것들은 그냥 개별로 처리)**

In [ ]:
## 영양소섭취
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축

## 차량
# B002-C003-0006:방향제(차량용)

## 아웃도어
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]

## 기타
# B002-C003-0008:[(향수), 로션, 세탁용세제]

## 설거지
# B002-C003-0009:수세미
# B002-C003-0010:키친타월
# B002-C003-0029:키친타월
# B002-C003-0013:주방세제(시트형)
# B002-C003-0016:주방세제
# B002-C003-0017:과탄산소다/베이킹소다

## 옷장
# B002-C003-0011:제습제

## 청소용
# B002-C003-0012:청소용세제

## 세탁
# B002-C003-0014:섬유유연제
# B002-C003-0018:표백제
# B002-C003-0015:세탁세제
# B002-C003-0048:세탁세제(유아)

## 양치
# B002-C003-0019:가글
# B002-C003-0020:전동칫솔
# B002-C003-0021:치실/치간칫솔
# B002-C003-0022:치약
# B002-C003-0023:칫솔

## 위생용품
# B002-C003-0024:화장지(롤)
# B002-C003-0026:각티슈
# B002-C003-0027:생리대

## 위생용품2
# B002-C003-0028:성인용 기저귀

## 면도
# B002-C003-0030:면도기
# B002-C003-0031:면도거품
# B002-C003-0033:전동면도기

## 손씻기
# B002-C003-0032:헨드케어(손비누)

## 목욕
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0039:헤어스타일링
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0040:에센스/트리트먼트

## 유아기저귀
# B002-C003-0025:물티슈
# B002-C003-0041:기저귀

## 유아분유
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0051:일반분유

## 유아 간식
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식

## 유아 목욕
# B002-C003-0045:유아용 스킨케어
# B002-C003-0046:유아바디&샴푸
# B002-C003-0047:유아선크림
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어

# **품목 재구매: 빈번하다/보통/빈번치않다**

In [ ]:
## 빈번하다 (생활필수품?)
# B002-C003-0010:키친타월
# B002-C003-0013:주방세제(시트형)
# B002-C003-0014:섬유유연제
# B002-C003-0015:세탁세제
# B002-C003-0016:주방세제
# B002-C003-0019:가글
# B002-C003-0021:치실/치간칫솔
# B002-C003-0022:치약
# B002-C003-0024:화장지(롤)
# B002-C003-0025:물티슈
# B002-C003-0026:각티슈
# B002-C003-0027:생리대
# B002-C003-0028:성인용 기저귀
# B002-C003-0029:키친타월
# B002-C003-0031:면도거품
# B002-C003-0032:헨드케어(손비누)
# B002-C003-0034:바디워시
# B002-C003-0035:샴푸
# B002-C003-0041:기저귀
# B002-C003-0046:유아바디&샴푸
# B002-C003-0048:세탁세제(유아)
# B002-C003-0051:일반분유


## 보통
# B002-C003-0001:뷰티/다이어트
# B002-C003-0002:비타민
# B002-C003-0003:기능별영양제
# B002-C003-0004:단백질
# B002-C003-0005:홍삼농축
# B002-C003-0006:방향제(차량용)
# B002-C003-0007:[자외선차단제, 영양제, (램프), 벌레퇴치제]
# B002-C003-0008:[(향수), 로션, 세탁용세제]
# B002-C003-0009:수세미
# B002-C003-0011:제습제
# B002-C003-0012:청소용세제
# B002-C003-0017:과탄산소다/베이킹소다
# B002-C003-0018:표백제
# B002-C003-0023:칫솔
# B002-C003-0036:에센스/트리트먼트
# B002-C003-0037:컨디셔너(린스)
# B002-C003-0038:에센스/트리트먼트
# B002-C003-0039:헤어스타일링
# B002-C003-0040:에센스/트리트먼트
# B002-C003-0045:유아용 스킨케어
# B002-C003-0047:유아선크림
# B002-C003-0049:유아칫솔
# B002-C003-0050:유아용 스킨케어
# B002-C003-0052:유아주스/차/떡벙(간식)
# B002-C003-0053:이유식


## 빈번치않다
# B002-C003-0020:전동칫솔
# B002-C003-0030:면도기
# B002-C003-0033:전동면도기
# B002-C003-0042:젖병소독기
# B002-C003-0043:젖병
# B002-C003-0044:유아식기(보관용기/식판/식기/컵/빨대)

# **train에 분류 구현**

In [ ]:
def fix_form(df):
    df['함께사용'] = 0
    df['소비주기'] = 0
    df['판매양상'] = 0

    selected_columns = ['ID', '제품', '대분류', '중분류', '소분류', '브랜드', '함께사용', '소비주기', '판매양상']
    selected_df = df[selected_columns]

    other_columns = [col for col in df.columns if col not in selected_columns]
    df = pd.concat([selected_df, df[other_columns]], axis=1)

    return df

In [ ]:
train_insert = fix_form(train)

In [ ]:
# 함께사용
target_subcategories = {
    'B002-C003-0001': 'e_nutrients', # 영양소섭취
    'B002-C003-0002': 'e_nutrients',
    'B002-C003-0003': 'e_nutrients',
    'B002-C003-0004': 'e_nutrients',
    'B002-C003-0005': 'e_nutrients',

    'B002-C003-0006': 'vehicle_freshener', # 차량방향제

    'B002-C003-0007': 'outdoor',

    'B002-C003-0008': 'etc',

    'B002-C003-0009': 'dish_wash',
    'B002-C003-0010': 'dish_wash',
    'B002-C003-0029': 'dish_wash',
    'B002-C003-0013': 'dish_wash',
    'B002-C003-0016': 'dish_wash',
    'B002-C003-0017': 'dish_wash',

    'B002-C003-0011': 'dehumidify', # 옷장 제습제

    'B002-C003-0012': 'for_cleaning',

    'B002-C003-0014': 'laundry',
    'B002-C003-0015': 'laundry',
    'B002-C003-0018': 'laundry',
    'B002-C003-0048': 'laundry',

    'B002-C003-0019': 'teeth',
    'B002-C003-0020': 'teeth',
    'B002-C003-0021': 'teeth',
    'B002-C003-0022': 'teeth',
    'B002-C003-0023': 'teeth',

    'B002-C003-0024': 'hygiene_products', # 위생용품
    'B002-C003-0026': 'hygiene_products',
    'B002-C003-0027': 'hygiene_products',

    'B002-C003-0028': 'hygiene_products_2',

    'B002-C003-0030': 'shaving',
    'B002-C003-0031': 'shaving',
    'B002-C003-0033': 'shaving',

    'B002-C003-0032': 'handwashing',

    'B002-C003-0034': 'bath',
    'B002-C003-0035': 'bath',
    'B002-C003-0037': 'bath',
    'B002-C003-0039': 'bath',
    'B002-C003-0036': 'bath',
    'B002-C003-0038': 'bath',
    'B002-C003-0040': 'bath',

    'B002-C003-0025': 'child_diaper',
    'B002-C003-0041': 'child_diaper',

    'B002-C003-0042': 'milk_powder',
    'B002-C003-0043': 'milk_powder',
    'B002-C003-0051': 'milk_powder',

    'B002-C003-0044': 'child_snack',
    'B002-C003-0052': 'child_snack',
    'B002-C003-0053': 'child_snack',

    'B002-C003-0045': 'child_bath',
    'B002-C003-0046': 'child_bath',
    'B002-C003-0047': 'child_bath',
    'B002-C003-0049': 'child_bath',
    'B002-C003-0050': 'child_bath'
}

for target_subcategory, evaluation in target_subcategories.items():
    rows_to_update = train_insert['소분류'] == target_subcategory
    train_insert.loc[rows_to_update, '함께사용'] = evaluation

In [ ]:
# 소비주기
target_subcategories = {
    'B002-C003-0010': 'frequent',
    'B002-C003-0013': 'frequent',
    'B002-C003-0014': 'frequent',
    'B002-C003-0015': 'frequent',
    'B002-C003-0016': 'frequent',
    'B002-C003-0019': 'frequent',
    'B002-C003-0021': 'frequent',
    'B002-C003-0022': 'frequent',
    'B002-C003-0024': 'frequent',
    'B002-C003-0025': 'frequent',
    'B002-C003-0026': 'frequent',
    'B002-C003-0027': 'frequent',
    'B002-C003-0028': 'frequent',
    'B002-C003-0029': 'frequent',
    'B002-C003-0031': 'frequent',
    'B002-C003-0032': 'frequent',
    'B002-C003-0034': 'frequent',
    'B002-C003-0035': 'frequent',
    'B002-C003-0041': 'frequent',
    'B002-C003-0046': 'frequent',
    'B002-C003-0048': 'frequent',
    'B002-C003-0051': 'frequent',

    'B002-C003-0001': 'normal',
    'B002-C003-0002': 'normal',
    'B002-C003-0003': 'normal',
    'B002-C003-0004': 'normal',
    'B002-C003-0005': 'normal',
    'B002-C003-0006': 'normal',
    'B002-C003-0007': 'normal',
    'B002-C003-0008': 'normal',
    'B002-C003-0009': 'normal',
    'B002-C003-0011': 'normal',
    'B002-C003-0012': 'normal',
    'B002-C003-0017': 'normal',
    'B002-C003-0018': 'normal',
    'B002-C003-0023': 'normal',
    'B002-C003-0036': 'normal',
    'B002-C003-0037': 'normal',
    'B002-C003-0038': 'normal',
    'B002-C003-0039': 'normal',
    'B002-C003-0040': 'normal',
    'B002-C003-0045': 'normal',
    'B002-C003-0047': 'normal',
    'B002-C003-0049': 'normal',
    'B002-C003-0050': 'normal',
    'B002-C003-0052': 'normal',
    'B002-C003-0053': 'normal',

    'B002-C003-0020': 'not_frequent',
    'B002-C003-0030': 'not_frequent',
    'B002-C003-0033': 'not_frequent',
    'B002-C003-0042': 'not_frequent',
    'B002-C003-0043': 'not_frequent',
    'B002-C003-0044': 'not_frequent'
}

for target_subcategory, evaluation in target_subcategories.items():
    rows_to_update = train_insert['소분류'] == target_subcategory
    train_insert.loc[rows_to_update, '소비주기'] = evaluation

In [ ]:
# 열별로 0이 있는지 확인
zero_check = train_insert.iloc[:, 6:8].eq(0).any()

# 결과 출력
print(zero_check)

함께사용    False
소비주기    False
dtype: bool


In [ ]:
# train_insert

In [ ]:
# train_insert.to_csv('./train_inserted.csv', index=False)

# **데이터 미리 처리 (함께사용, 소비주기)**

In [ ]:
train_insert.drop(['ID', '제품', '대분류', '중분류', '소분류', '브랜드'], axis=1, inplace=True) # 4개의 상태 특성을 뺌.

In [ ]:
from sklearn.preprocessing import LabelEncoder

def label_encoding(data):
    label_encoder = LabelEncoder()
    categorical_columns = ['함께사용', '소비주기']

    for col in categorical_columns:
        label_encoder.fit(data[col])
        data[col] = label_encoder.transform(data[col])

    return data

In [ ]:
label_encoding(train_insert)
print('Done.')

Done.


In [ ]:
train_insert

,함께사용,소비주기,판매양상,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,3,2,0,0,2,0
2,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,6,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,13,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15886,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,0,2,4,1,1,3
15887,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15888,3,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2


# **처리 완료된 코드**

In [ ]:
함께사용_data = train_insert.copy()
함께사용_data.iloc[:, 3:] = 0
함께사용_data.drop(['소비주기', '판매양상'], axis=1, inplace=True)
for i in tqdm(range(len(함께사용_data))):
  함께사용_data.iloc[i, 1:] = 함께사용_data.iloc[i, 0]
함께사용_data.drop('함께사용', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
소비주기_data = train_insert.copy()
소비주기_data.iloc[:, 3:] = 0
소비주기_data.drop(['함께사용', '판매양상'], axis=1, inplace=True)
for i in tqdm(range(len(소비주기_data))):
  소비주기_data.iloc[i, 1:] = 소비주기_data.iloc[i, 0]
소비주기_data.drop('소비주기', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
판매양상_data = train_insert.copy()
판매양상_data.iloc[:, 3:] = 0
판매양상_data.drop(['함께사용', '소비주기'], axis=1, inplace=True)
for i in tqdm(range(len(판매양상_data))):
  판매양상_data.iloc[i, 1:] = 판매양상_data.iloc[i, 0]
판매양상_data.drop('판매양상', axis=1, inplace=True)

  0%|          | 0/15890 [00:00<?, ?it/s]

In [ ]:
함께사용_data.iloc[:, 0].unique()

array([ 0,  3,  6,  9, 16,  2,  8,  4,  5, 17, 14,  1, 12,  7, 10, 11, 15,
       13])

In [ ]:
소비주기_data.iloc[:, 0].unique()

array([1, 2, 0])

In [ ]:
판매양상_data.iloc[:, 0].unique()

array([2, 1, 3, 0])

In [ ]:
# 함께사용_data.to_csv('./함께사용_4.csv', index=False)
# 소비주기_data.to_csv('./소비주기_4.csv', index=False)
# 판매양상_data.to_csv('./판매양상.csv', index=False)

In [ ]:
판매양상_data

,2022-01-01,2022-01-02,2022-01-03,2022-01-04,2022-01-05,2022-01-06,2022-01-07,2022-01-08,2022-01-09,2022-01-10,2022-01-11,2022-01-12,2022-01-13,2022-01-14,2022-01-15,2022-01-16,2022-01-17,2022-01-18,2022-01-19,2022-01-20,2022-01-21,2022-01-22,2022-01-23,2022-01-24,2022-01-25,...,2023-03-11,2023-03-12,2023-03-13,2023-03-14,2023-03-15,2023-03-16,2023-03-17,2023-03-18,2023-03-19,2023-03-20,2023-03-21,2023-03-22,2023-03-23,2023-03-24,2023-03-25,2023-03-26,2023-03-27,2023-03-28,2023-03-29,2023-03-30,2023-03-31,2023-04-01,2023-04-02,2023-04-03,2023-04-04
0,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
4,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15885,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
15886,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
15887,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
15888,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


# **저장 history**

In [ ]:
# 함께사용_data.to_csv('./함께사용_3.csv', index=False)
# 소비주기_data.to_csv('./소비주기_3.csv', index=False)
# 생건몰분류_data.to_csv('./생건몰분류_2.csv', index=False)

In [ ]:
# 함께사용_data.to_csv('./함께사용_2.csv', index=False)
# 소비주기_data.to_csv('./소비주기_2.csv', index=False)
# 함께사용_data.to_csv('./함께사용.csv', index=False)
# 소비주기_data.to_csv('./소비주기.csv', index=False)

In [ ]:
# 생건몰분류_data.to_csv('./생건몰분류.csv', index=False)
# 비슷한제품_data.to_csv('./비슷한제품.csv', index=False)